In [364]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/csv/morg08.csv")
df.head()

,hhid,intmonth,hurespli,hrhtype,minsamp,hrlonglk,hrsample,hrhhid2,serial,hhnum,...,ym_file,ym,ch02,ch35,ch613,ch1417,ch05,ihigrdc,docc00,dind02
0,2600310997690,1,2.0,1,8,2,8200,82001,1,1,...,576,561,0.0,1.0,1.0,0.0,1.0,12.0,19.0,4.0
1,2600310997690,1,2.0,1,8,2,8200,82001,1,1,...,576,561,0.0,1.0,1.0,0.0,1.0,18.0,10.0,42.0
2,7087707096191,1,2.0,1,4,2,8300,83001,1,1,...,576,573,0.0,0.0,1.0,0.0,0.0,18.0,1.0,24.0
3,7087707096191,1,2.0,1,4,2,8300,83001,1,1,...,576,573,0.0,0.0,1.0,0.0,0.0,18.0,NaN,NaN
4,41110310970391,1,1.0,1,8,2,8200,82001,1,1,...,576,561,0.0,0.0,0.0,0.0,0.0,16.0,1.0,36.0


In [365]:
#why does uhourse have a -4??
df.value_counts('uhourse')
df = df[df['uhourse'] > 0]
#hadtouseuhourse>0toavoidhourylwagebeinginfinity

In [366]:
#Creating hourly_wage
# HOW SHOULD I TREAT NANS IN HOURLY WAGE
new_cols = pd.DataFrame({
    'hourly_wage': df['earnwke'] / df['uhourse']})
df = pd.concat([df, new_cols], axis=1)

In [367]:
#Adding log and age cutoff variables
new_cols = pd.DataFrame({'log_hourly': np.log(df["hourly_wage"]), 'age_le_30': (df["age"] <= 30), 'age_ge_55': (df["age"] >= 55)})
df = pd.concat([df, new_cols], axis=1)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [368]:
#recoding sex variable
new_col = pd.DataFrame({'female': (df['sex'] == 2).astype(int)})
df = pd.concat([df, new_col], axis=1)


The `grade92` variable tells us highest grade completed. I created dummy variables as follows:
cateogry 1: 38 = high school no diploma
cateogry 2: 39 = hs diploma, 40 = some college but no degree
cateogry 3: 43 = bachelor's degree, 41 = associate degree occupational/vocational, 42 = associate degree academic program
category 4: 44 and above are different types of post-college degrees

In [369]:
#Recoding educational cateogories

new_cols = pd.DataFrame({
    'no_hs_diploma': (df['grade92'] <= 38).astype(int),
    'hs_some_college': df['grade92'].isin([39, 40]).astype(int),
    'college': df['grade92'].isin([41, 42, 43]).astype(int),
    'post_college': (df['grade92'] >= 44).astype(int)
})

df = pd.concat([df, new_cols], axis=1)

The codebook explains how to calculate hourly wages, which I did in the first cell: "Earnings are collected per hour for hourly workers, and per week for other workers. If you want a consistent hourly wage series during entire period, you should use earnwke/uhourse. This gives imputed hourly wage for weekly workers and actual hourly wage for hourly workers. But check earnwke for top-coding. Do not use any wage data that may be present for self-employed workers"

In [370]:
smaller_df = df[['hhid', 'state', 'county', 'class94', 'earnhre', 'age', 'female', 'grade92', 'unioncov', 'uhourse', 'weight', 'hourly_wage', 'log_hourly', 'age_le_30', 'age_ge_55', 'no_hs_diploma', 'hs_some_college', 'college', 'post_college']]
smaller_df.head()

,hhid,state,county,class94,earnhre,age,female,grade92,unioncov,uhourse,weight,hourly_wage,log_hourly,age_le_30,age_ge_55,no_hs_diploma,hs_some_college,college,post_college
0,2600310997690,63,0,4.0,2100.0,41,0,39,2.0,40.0,2846.6161,21.00000,3.044522,False,False,0,1,0,0
1,2600310997690,63,0,5.0,2100.0,40,1,44,2.0,37.0,3118.6074,21.00000,3.044522,False,False,0,0,0,1
2,7087707096191,63,0,4.0,NaN,40,0,44,2.0,40.0,3719.7807,40.86525,3.710280,False,False,0,0,0,1
4,41110310970391,63,73,6.0,NaN,68,0,43,NaN,30.0,3554.8098,NaN,NaN,False,True,0,0,1,0
5,41110310970391,63,73,1.0,1450.0,63,1,43,2.0,40.0,4511.5217,23.55750,3.159444,False,True,0,0,1,0


In [371]:
#Illinois = state == 33
illinois_df = smaller_df[smaller_df['state'] == 33]
illinois_df

,hhid,state,county,class94,earnhre,age,female,grade92,unioncov,uhourse,weight,hourly_wage,log_hourly,age_le_30,age_ge_55,no_hs_diploma,hs_some_college,college,post_college
7643,18351370993090,33,0,5.0,NaN,23,1,39,NaN,40.0,3775.5995,21.634500,3.074289,True,False,0,1,0,0
7644,31691370940990,33,0,5.0,NaN,26,0,43,2.0,40.0,4153.6278,9.807500,2.283147,True,False,0,0,1,0
7646,33341370924585,33,0,6.0,NaN,47,0,42,NaN,40.0,2833.8796,NaN,NaN,False,False,0,0,1,0
7647,34371370972794,33,0,4.0,NaN,41,0,43,2.0,40.0,3798.1489,18.269000,2.905206,False,False,0,0,1,0
7648,34371370972794,33,0,4.0,NaN,40,1,43,2.0,40.0,3261.2051,28.846000,3.361971,False,False,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299347,925191769700967,33,0,4.0,NaN,36,1,39,2.0,30.0,3083.8906,15.366667,2.732201,False,False,0,1,0,0
299348,967193249700939,33,113,2.0,NaN,41,1,46,2.0,40.0,2771.0185,25.961500,3.256615,False,False,0,0,0,1
299349,967193249700939,33,113,1.0,1075.0,42,0,40,NaN,60.0,2837.2760,12.910000,2.558002,False,False,0,1,0,0
299351,976047815129860,33,115,7.0,NaN,54,1,40,NaN,17.0,2723.0936,NaN,NaN,False,False,0,1,0,0


In [373]:
illinois_df['hourly_wage'].isna().sum()

np.int64(453)

In [344]:
illinois_df['earnhre'].isna().sum()

np.int64(2701)

In [335]:
illinois_df['unioncov'].isna().sum()

np.int64(1318)

In [336]:
illinois_df['uhourse'].isna().sum()

np.int64(0)

Government jobs (federal, state, local) are codes 1, 2, and 3.
Private, for profit jobs are 4.
Private, non-profit jobs are 5.

In [374]:
#Creating IL public df

illinois_public_df = illinois_df[illinois_df['class94'].isin([1, 2, 3])]

In [359]:
#Creating IL private df
illinois_private_df = illinois_df[illinois_df['class94'].isin([4, 5])]

In [376]:
#Creating other states public df
other_states_df = smaller_df[smaller_df['state'] != 33]
public_other_states_df = other_states_df[other_states_df['class94'].isin([1, 2, 3])]


In [377]:
#new df to make Table 1
columns=['IL_pub', 'IL_priv', 'other_pub', 'pval_ILpubvpriv', 'pval_ILvotherpub']
table_1_df = pd.DataFrame(columns = columns)

hourly_wage = [illinois_public_df['hourly_wage'].mean(), illinois_private_df['hourly_wage'].mean(), other_states_df['hourly_wage'].mean()]
log_hourly_wage = [illinois_public_df['log_hourly'].mean(), illinois_private_df['log_hourly'].mean(), other_states_df['log_hourly'].mean()]
age = [illinois_public_df['age'].mean(), illinois_private_df['age'].mean(), other_states_df['age'].mean()]
share_age_le30 = [illinois_public_df['age_le_30'].mean(), illinois_private_df['age_le_30'].mean(), other_states_df['age_le_30'].mean()]
share_age_ge55 = [illinois_public_df['age_ge_55'].mean(), illinois_private_df['age_ge_55'].mean(), other_states_df['age_ge_55'].mean()]
female = [illinois_public_df['female'].mean(), illinois_private_df['female'].mean(), other_states_df['female'].mean()]
no_hs_diploma = [illinois_public_df['no_hs_diploma'].mean(), illinois_private_df['no_hs_diploma'].mean(), other_states_df['no_hs_diploma'].mean()]
hs_some_college = [illinois_public_df['hs_some_college'].mean(), illinois_private_df['hs_some_college'].mean(), other_states_df['hs_some_college'].mean()]
college = [illinois_public_df['hs_some_college'].mean(), illinois_private_df['hs_some_college'].mean(), other_states_df['hs_some_college'].mean()]
college = [illinois_public_df['college'].mean(), illinois_private_df['college'].mean(), other_states_df['college'].mean()]
post_college = [illinois_public_df['post_college'].mean(), illinois_private_df['post_college'].mean(), other_states_df['post_college'].mean()]
weekly_hours = [illinois_public_df['uhourse'].mean(), illinois_private_df['uhourse'].mean(), other_states_df['uhourse'].mean()]
union_cov = [illinois_public_df['unioncov'].mean(), illinois_private_df['unioncov'].mean(), other_states_df['unioncov'].mean()]
n = [len(illinois_public_df), len(illinois_private_df), len(other_states_df)]


rows = ['Hourly Wage', 'Log Hourly Wage', 'Age', 'Share Age ≤ 30', 'Share Age ≥ 55',
        'Female Share', 'No HS Diploma', 'HS Some College', 'College', 'Post College',
        'Usual Weekly Hours', 'Union Coverage', 'N (Unweighted)']

all_values = [
    hourly_wage,
    log_hourly_wage,
    age,
    share_age_le30,
    share_age_ge55,
    female,
    no_hs_diploma,
    hs_some_college,
    college,
    post_college,
    weekly_hours,
    union_cov,
    n
]



#don't forget to do p values and add to table!!

In [ ]:
#How should i deal with NA?????
#ADD COMMENTS TO EVERYTHING (OR MARKDOWN)

#do the log hourly and etc up at the df level not for each